Nama   : Roland Albertian Sehapikang  
Kelas  : IF403  
NIM    : 240401010249  

In [8]:
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']


transaksi = []
for i in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))


for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [10]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("5 Baris Pertama Matriks Transaksi (One-Hot):")
print(df.head())

5 Baris Pertama Matriks Transaksi (One-Hot):
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [11]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print("\nTop 10 Frequent Itemset:")
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemset:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [12]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print("Aturan Asosiasi Hasil Filtrasi (Top 10 berdasarkan Lift):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Aturan Asosiasi Hasil Filtrasi (Top 10 berdasarkan Lift):
         antecedents consequents  support  confidence      lift
9        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
13  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
12      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
15     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
10     (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
14   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


**1.**   **Aturan mana yang paling kuat (Lift tertinggi)?**  
Aturan terkuat dipimpin oleh kombinasi **{Selai}** $\rightarrow$ **{Roti}** atau **{Roti}** $\rightarrow$ **{Selai}** (bergantung pada hasil randomisasi seed tepatnya, namun secara konsisten pola Roti dan Selai mendominasi).  

**2.**   **Apakah masuk akal secara bisnis?**  
Ya, sangat masuk akal secara bisnis karena produk tersebut merupakan barang komplementer (saling melengkapi) yang biasa dikonsumsi bersamaan untuk sarapan.





In [13]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy', 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx]
    return katalog.iloc[[i for i, _ in skor[:top_n]]]['produk'].tolist()

print('Rekomendasi Content-Based mirip dengan Roti:', rekomendasi_serupa('Roti'))

Rekomendasi Content-Based mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [14]:
produk_target = 'Roti'

# Rekomendasi dari Asociation Rules
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('=== Rekomendasi dari Association Rules ===')
print(rules_terkait[['consequents', 'lift']].head())

print('\n=== Rekomendasi dari Content-Based ===')
print(rekomendasi_serupa(produk_target))

=== Rekomendasi dari Association Rules ===
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115

=== Rekomendasi dari Content-Based ===
['Selai', 'Sereal', 'Susu']




**1.   Apakah kedua pendekatan memberi rekomendasi yang konsisten?**  
Tidak selalu sama secara item, tetapi memiliki karakteristik unik. Association Rules merekomendasikan **Selai** karena data transaksi riil membuktikan keduanya sering dibeli bersama (cross-commodity relationship). Sementara Content-Based Filtering merekomendasikan produk dalam satu kategori Bakery seperti yang ada pada daftar katalog.  
**2.   Kapan sebaiknya menggunakan salah satu atau digabungkan (Hybrid)?**  
*   **Association Rules:** Sangat cocok digunakan untuk penataan tata letak produk di toko fisik (layouting), pembuatan paket promosi (bundling), atau e-commerce skala dasar.
*   **Content-Based Filtering:** Tepat digunakan saat sistem menghadapi masalah cold start (pengguna baru yang belum memiliki transaksi), karena sistem hanya bergantung pada metadata produk.
*   **Hybrid System:** Merupakan pilihan terbaik untuk industri skala produksi modern (seperti e-commerce besar). Dengan menggabungkannya, sistem dapat menyajikan rekomendasi yang kaya ragam (serendipity) dari sisi transaksi kolektif, sekaligus menjaga relevansi fungsi dari sisi kemiripan deskripsi barang.





